# Resume Screening RAG Pipeline (cleaned)

Fixes applied vs. the original notebook:

1. Every function is defined exactly once, in dependency order (no more `NameError: exact_skill_match not defined` from calling something before a later cell defines it).
2. Requirement normalization is consistent and case-insensitive everywhere (single `ALIASES` dict + `normalize_requirement()`).
3. `exact_skill_match()` uses word-boundary regex instead of bare substring matching, so "SQL" no longer accidentally matches inside "MySQL" for the wrong reasons (still matches MySQL, but explicitly via an allowlist, not by accident).
4. `evaluate_retrieval`'s dead `item == "..."` comparison (always False) is removed.
5. Ollama response access is consistent (dict-style) everywhere.
6. No reliance on leftover global loop variables.

## Setup

In [1]:
import re
import json
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import ollama

In [2]:
MODEL_NAME = "llama3.2"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHROMA_PATH = "../dataset/chroma"
COLLECTION_NAME = "resume_chunk_v2"
RERANK_THRESHOLD = -10.3  # tune per your reranker/data -- see note at the bottom

SECTION_NAMES = ["ABOUT ME", "SKILLS", "PROFESSIONAL EXPERIENCE", "PROJECTS", "EDUCATION"]

# Single, case-insensitive alias table used everywhere (fixes the old
# case-sensitive SKILL_NORMALIZATION dict that rarely matched).
ALIASES = {
    "python programming": "Python",
    "python programming skills": "Python",
    "sql databases": "SQL",
    "sql database": "SQL",
    "sql/database knowledge": "SQL",
    "experience working with git and github": "Git and GitHub",
    "good problem solving skills": "Problem solving",
}

## 1. PDF extraction + cleaning

In [3]:
def extract_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


def clean_text(text: str) -> str:
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text

## 2. Section-aware chunking

In [4]:
def extract_sections(text: str, section_names=SECTION_NAMES) -> dict:
    pattern = r"(?m)^(" + "|".join(map(re.escape, section_names)) + r")\s*$"
    parts = re.split(pattern, text, flags=re.IGNORECASE)

    sections = {}
    current_section = None
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if part.upper() in section_names:
            current_section = part.upper()
            sections[current_section] = ""
        elif current_section:
            sections[current_section] += " " + part
    return sections


def chunk_skills(content: str) -> list:
    """
    Split SKILLS into individual subsections.
    """

    subsection_patterns = {
        "languages": r"Languages:\s*(.*?)(?=Frontend:|Backend:|Database:|Tools:|Soft Skills:|Communication Skills:|$)",
        "frontend": r"Frontend:\s*(.*?)(?=Backend:|Database:|Tools:|Soft Skills:|Communication Skills:|$)",
        "backend": r"Backend:\s*(.*?)(?=Database:|Tools:|Soft Skills:|Communication Skills:|$)",
        "database": r"Database:\s*(.*?)(?=Tools:|Soft Skills:|Communication Skills:|$)",
        "tools": r"Tools:\s*(.*?)(?=Soft Skills:|Communication Skills:|$)",
        "soft_skills": r"Soft Skills:\s*(.*?)(?=Communication Skills:|$)",
        "communication": r"Communication Skills:\s*(.*?)$",
    }

    chunks = []

    for subsection, pattern in subsection_patterns.items():

        match = re.search(
            pattern,
            content,
            flags=re.IGNORECASE | re.DOTALL
        )

        if not match:
            continue

        text = match.group(1).strip()

        if not text:
            continue

        chunks.append({
            "text": f"SKILLS — {subsection.upper()}\n{text}",
            "section": "SKILLS",
            "subsection": subsection,
        })

    return chunks


def chunk_section(
    text: str,
    section: str,
    max_words: int = 150,
    overlap: int = 30
) -> list:

    if section == "SKILLS":
        return chunk_skills(text)

    words = text.split()

    if len(words) <= max_words:
        return [{
            "text": f"{section}\n{text}",
            "section": section
        }]

    chunks = []

    start = 0

    while start < len(words):

        end = start + max_words

        piece = " ".join(words[start:end])

        chunks.append({
            "text": f"{section}\n{piece}",
            "section": section
        })

        start += max_words - overlap

    return chunks

## 3. Embedding + vector store

In [5]:
_embed_model = None
_reranker = None
_collection = None


def get_embed_model() -> SentenceTransformer:
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return _embed_model


def get_reranker() -> CrossEncoder:
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder(RERANK_MODEL_NAME)
    return _reranker


def get_collection():
    global _collection
    if _collection is None:
        client = chromadb.PersistentClient(path=CHROMA_PATH)
        _collection = client.get_or_create_collection(name=COLLECTION_NAME)
    return _collection


def build_metadata(chunk: dict, resume_id: str, candidate_id: str) -> dict:
    metadata = {"resume_id": resume_id, "candidate_id": candidate_id, "section": chunk["section"]}
    for key in ("subsection", "project", "job_title", "company"):
        if key in chunk:
            metadata[key] = chunk[key]
    return metadata


def index_chunks(chunks: list, resume_id: str = "resume_001", candidate_id: str = "candidate_001"):
    """Embed a list of {"text": ..., "section": ..., ...} chunks and add to Chroma."""
    model = get_embed_model()
    collection = get_collection()

    documents = [c["text"] for c in chunks]
    embeddings = model.encode(documents)
    ids = [f"{resume_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [build_metadata(c, resume_id, candidate_id) for c in chunks]

    collection.upsert(ids=ids, documents=documents, embeddings=embeddings.tolist(), metadatas=metadatas)
    return len(chunks)


def reset_collection():

    global _collection

    client = chromadb.PersistentClient(path=CHROMA_PATH)

    try:
        client.delete_collection(name=COLLECTION_NAME)
        print("Old collection deleted.")
    except Exception:
        print("No old collection found.")

    _collection = client.get_or_create_collection(
        name=COLLECTION_NAME
    )

    print("New collection created.")

    return _collection

## 4. Retrieval + section routing + reranking

In [6]:
def search_resume(query: str, n_results: int = 3, section: str = None) -> dict:
    model = get_embed_model()
    collection = get_collection()

    query_embedding = model.encode(query).tolist()
    kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"],
    }
    if section:
        kwargs["where"] = {"section": section}

    return collection.query(**kwargs)


def detect_section(query: str):
    q = query.lower()
    if any(w in q for w in ["project", "projects", "built", "application", "app"]):
        return "PROJECTS"
    if any(w in q for w in ["work", "worked", "experience", "job", "company", "internship"]):
        return "PROFESSIONAL EXPERIENCE"
    if any(w in q for w in ["education", "study", "studied", "college", "school", "degree", "training"]):
        return "EDUCATION"
    if any(w in q for w in ["skill", "skills", "know", "database", "language", "framework", "tool"]):
        return "SKILLS"
    return None


def rerank_results(query: str, results: dict) -> list:
    documents = results["documents"][0]
    if not documents:
        return []

    reranker = get_reranker()
    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs)

    ranked = [
        {
            "document": documents[i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
            "rerank_score": float(scores[i]),
        }
        for i in range(len(documents))
    ]
    ranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return ranked


def retrieve_and_rerank(query: str, n_results: int = 10, max_results: int = 3,
                         threshold: float = RERANK_THRESHOLD) -> list:
    section = detect_section(query)
    results = search_resume(query=query, n_results=n_results, section=section)
    ranked = rerank_results(query, results)
    relevant = [r for r in ranked if r["rerank_score"] >= threshold]
    return relevant[:max_results]

## 5. Skill verification

`exact_skill_match` uses word-boundary regex instead of bare substring matching. Plain substring matching would match `"Java"` inside `"JavaScript"`, which is wrong. `"SQL"` matching inside `"MySQL"` is kept, but as an explicit, intentional rule via `SUBSTRING_OK` rather than an accident.

In [7]:
# Terms where a substring match against a longer word is intentionally OK.
# Left empty on purpose: "MySQL" should NOT count as evidence for "SQL"
# unless the resume literally contains the word "SQL" somewhere. Any
# substring allowlist here re-introduces that false-positive risk.
SUBSTRING_OK = {}


def normalize_requirement(requirement: str) -> str:
    key = requirement.strip().lower()
    return ALIASES.get(key, requirement)


def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#.\- ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _term_matches(term: str, evidence: str) -> bool:
    if re.search(rf"\b{re.escape(term)}\b", evidence):
        return True
    for allowed_word in SUBSTRING_OK.get(term, []):
        if allowed_word in evidence:
            return True
    return False


def exact_skill_match(skill: str, evidence_text: str) -> bool:
    skill_terms = re.findall(r"[a-z0-9+#.-]+", skill.lower())
    evidence_norm = normalize_text(evidence_text)
    return all(_term_matches(term, evidence_norm) for term in skill_terms)


def verify_skill_llm(skill: str, evidence_text: str) -> bool:

    prompt = f"""
You are a strict resume evidence verifier.

Requirement:
{skill}

Resume evidence:
{evidence_text}

Determine whether the resume evidence explicitly supports
the requirement.

Rules:

1. Return TRUE only when the requirement itself is explicitly
   mentioned or the evidence is an unmistakable direct statement
   of experience with that requirement.

2. Do NOT infer related skills.

3. Python does NOT imply machine learning.

4. Python does NOT imply scikit-learn.

5. MySQL does NOT imply SQL unless SQL is explicitly mentioned.

6. Git does NOT imply GitHub.

7. Transformers do NOT imply NLP.

8. A general programming language does not imply a framework,
   library, algorithm, or domain.

9. If uncertain, return FALSE.

Return ONLY:
TRUE
or
FALSE
"""
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Return only TRUE or FALSE."},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0},
    )
    answer = response["message"]["content"].strip().upper()
    return answer == "TRUE"


def select_best_evidence(evidence, max_results=2):
    """
    Keep only the strongest verified evidence.
    Prefer exact/stronger reranker matches.

    Without this, check_skill can return every chunk that happened to pass
    verification (e.g. ABOUT ME, SKILLS, and PROJECTS all mentioning
    "frontend"), which makes reports noisy and non-deterministic in ordering.
    """
    if not evidence:
        return []

    sorted_evidence = sorted(
        evidence,
        key=lambda x: x.get("rerank_score", float("-inf")),
        reverse=True
    )

    return sorted_evidence[:max_results]


def check_skill(skill: str) -> dict:
    original_skill = skill
    normalized_skill = normalize_requirement(skill)

    query = normalized_skill
    evidence = retrieve_and_rerank(query)

    if not evidence:
        return {"skill": original_skill, "normalized_skill": normalized_skill, "matched": False, "evidence": []}

    verified_evidence = []
    for item in evidence:
        document = item["document"]
        if exact_skill_match(normalized_skill, document) or verify_skill_llm(normalized_skill, document):
            verified_evidence.append(item)

    if not verified_evidence:
        return {"skill": original_skill, "normalized_skill": normalized_skill, "matched": False, "evidence": []}

    evidence = select_best_evidence(verified_evidence)

    return {
        "skill": original_skill,
        "normalized_skill": normalized_skill,
        "matched": True,
        "evidence": evidence
    }


## 6. Job requirement extraction + candidate scoring

In [8]:
def extract_job_requirements(job_description: str) -> dict:
    prompt = f"""
You are a job description analysis assistant.

Extract EVERY explicit requirement from the job description.

Return ONLY valid JSON with exactly this structure:

{{
    "skills": [],
    "tools": [],
    "frameworks": [],
    "databases": [],
    "soft_skills": []
}}

Rules:
- Extract EVERY explicit requirement, never infer or add implied ones.
- Do not merge related requirements (e.g. "PyTorch" and "TensorFlow" stay separate).
- soft_skills: only include skills explicitly stated, do not add common workplace skills.
- Do not add explanations outside the JSON.

JOB DESCRIPTION:
{job_description}
"""
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You extract structured job requirements. Return valid JSON only."},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0},
    )
    content = response["message"]["content"]
    return json.loads(content)


def flatten_requirements(requirements: dict) -> list:
    flat = []
    for items in requirements.values():
        for item in items:
            item = item.strip()
            if item:
                flat.append(item)
    return list(dict.fromkeys(flat))  # dedupe, keep order

In [9]:
def check_requirements(requirements):
    """
    Check every job requirement against verified resume evidence.
    Returns the raw check_skill() result for each requirement, so
    downstream consumers (report printing, prompt building, future
    per-candidate ranking) all work off the same shape.
    """
    results = []

    for requirement in requirements:
        result = check_skill(requirement)
        results.append(result)

    return results


def calculate_match_score(results: list) -> float:
    if not results:
        return 0.0

    matched = sum(
        result["matched"]
        for result in results
    )

    return round(
        matched / len(results) * 100,
        2
    )


def build_match_report(requirements: list) -> dict:
    results = check_requirements(requirements)

    score = calculate_match_score(results)

    matched = [
        result for result in results
        if result["matched"]
    ]

    missing = [
        result for result in results
        if not result["matched"]
    ]

    return {
        "score": score,
        "results": results,
        "matched": matched,
        "missing": missing
    }


# Kept as a thin alias so earlier cells that reference analyze_requirements
# (report printing, prompt building) keep working unchanged.
def analyze_requirements(requirements: list) -> dict:
    return build_match_report(requirements)


In [10]:
def print_candidate_report(analysis: dict):
    print("=" * 60)
    print("AI RESUME SCREENING REPORT")
    print("=" * 60)
    print(f"\nMATCH SCORE: {analysis['score']:.2f}%")

    print("\nMATCHED")
    print("-" * 60)
    for result in analysis["results"]:
        if not result["matched"]:
            continue
        print(f"\n\u2713 {result['skill']}")
        for evidence in result["evidence"]:
            section = evidence["metadata"].get("section")
            subsection = evidence["metadata"].get("subsection")
            print(f"  Source: {section} \u2192 {subsection}")
            print(f"  Evidence: {evidence['document']}")

    print("\nMISSING")
    print("-" * 60)
    for result in analysis["results"]:
        if result["matched"]:
            continue
        print(f"\n\u2717 {result['skill']}")
        print("  No supporting evidence found in the resume.")


def build_evidence_context(analysis: dict) -> str:
    grouped = {}
    for result in analysis["results"]:
        if not result["matched"]:
            continue
        skill = result["skill"]
        grouped.setdefault(skill, [])
        for evidence in result["evidence"]:
            metadata = evidence["metadata"]
            item = (metadata.get("section"), metadata.get("subsection"), evidence["document"])
            if item not in grouped[skill]:
                grouped[skill].append(item)

    parts = []
    for skill, evidence_list in grouped.items():
        block = f"Requirement: {skill}\n"
        for section, subsection, document in evidence_list:
            block += f"\nSource: {section} \u2192 {subsection}\nEvidence:\n{document}\n"
        parts.append(block)
    return "\n".join(parts)


def build_candidate_prompt(analysis: dict) -> str:

    matched = [
        r["skill"]
        for r in analysis["results"]
        if r["matched"]
    ]

    missing = [
        r["skill"]
        for r in analysis["results"]
        if not r["matched"]
    ]

    return f"""
Analyze ONLY these verified job requirements.

MATCH SCORE:
{analysis['score']:.2f}%

MATCHED REQUIREMENTS:
{json.dumps(matched)}

MISSING REQUIREMENTS:
{json.dumps(missing)}

Return ONLY this JSON structure:

{{
    "overall_assessment": "",
    "confirmed_strengths": [],
    "missing_requirements": [],
    "final_recommendation": ""
}}

Rules:

- confirmed_strengths MUST contain only matched requirements.
- missing_requirements MUST contain only missing requirements.
- Do not mention any other resume skill.
- Do not mention React unless it appears in MATCHED REQUIREMENTS.
- Do not mention Laravel unless it appears in MATCHED REQUIREMENTS.
- Python does not imply machine learning.
- Python does not imply scikit-learn.
- Python does not imply NLP.
- MySQL does not imply SQL.
- Transformers do not imply NLP.
- Missing evidence must remain missing.
- If uncertain, say that the requirement is not verified.
- Do not recommend another job role.
- Do not invent experience.

"""


def generate_candidate_assessment(prompt: str) -> dict:

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": """
You are a strict resume screening assistant.

Use ONLY the verified requirements provided by the user.

Return ONLY valid JSON.

Do not mention any skill that is not present in
MATCHED REQUIREMENTS or MISSING REQUIREMENTS.

Do not infer skills.

Do not invent experience.

Do not mention any technology, skill, experience,
or qualification unless it appears in the supplied
MATCHED REQUIREMENTS or MISSING REQUIREMENTS.
"""
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={"temperature": 0},
    )

    content = response["message"]["content"].strip()

    return json.loads(content)

def validate_assessment(assessment: dict, analysis: dict) -> dict:
    """
    Post-filter the LLM's JSON output against the verified match results.
    The matching engine remains the source of truth -- if the model listed
    a skill under confirmed_strengths that isn't actually in our matched
    set (or vice versa for missing_requirements), drop it rather than
    trusting the model's free-text instruction-following.
    """
    matched = {
        r["skill"]
        for r in analysis["results"]
        if r["matched"]
    }

    missing = {
        r["skill"]
        for r in analysis["results"]
        if not r["matched"]
    }

    assessment["confirmed_strengths"] = [
        skill
        for skill in assessment.get("confirmed_strengths", [])
        if skill in matched
    ]

    assessment["missing_requirements"] = [
        skill
        for skill in assessment.get("missing_requirements", [])
        if skill in missing
    ]

    return assessment


## 7. End-to-end run

In [11]:
raw_text = extract_pdf_text("../dataset/sefat_khan.pdf")
text = clean_text(raw_text)

sections = extract_sections(text)

chunks = []

for section, content in sections.items():
    chunks.extend(
        chunk_section(content, section)
    )

print("Chunks created:", len(chunks))

for i, chunk in enumerate(chunks):
    print(
        i,
        "| section =", chunk["section"],
        "| subsection =", chunk.get("subsection"),
        "|",
        chunk["text"]
    )

# IMPORTANT: remove old Chroma data
reset_collection()

# Index fresh chunks
index_chunks(chunks)

print("Indexed", len(chunks), "chunks")

Chunks created: 11
0 | section = ABOUT ME | subsection = None | ABOUT ME
 Full Stack Web Developer with hands-on industry experience building responsive and scalable web 
applications using React, Laravel, and MySQL. Skilled in frontend performance optimization, REST API 
integration, authentication systems, and modern UI development.
1 | section = SKILLS | subsection = languages | SKILLS — LANGUAGES
JavaScript, TypeScript, PHP, Python, SQL, HTML, CSS 

2 | section = SKILLS | subsection = frontend | SKILLS — FRONTEND
React.js, Redux Toolkit, Tailwind CSS, Bootstrap, React Bootstrap 

3 | section = SKILLS | subsection = backend | SKILLS — BACKEND
Laravel, REST APIs, Authentication Systems 

4 | section = SKILLS | subsection = database | SKILLS — DATABASE
MySQL, Firebase 

5 | section = SKILLS | subsection = tools | SKILLS — TOOLS
Git, GitHub 
 

6 | section = SKILLS | subsection = soft_skills | SKILLS — SOFT_SKILLS
Time management, problem solving, teamwork, adaptability, attention

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 11 chunks


In [12]:
job_description = """
Machine Learning Engineer Intern

Requirements:
- Strong Python programming skills
- Knowledge of machine learning algorithms
- Experience with scikit-learn
- Knowledge of NLP and Transformers
- Experience with PyTorch or TensorFlow
- Knowledge of SQL databases
- Good problem solving skills
- Experience working with Git and GitHub
"""

requirements = extract_job_requirements(job_description)
flat_requirements = flatten_requirements(requirements)
print(flat_requirements)

['Python programming', 'Machine learning algorithms', 'NLP', 'Transformers', 'scikit-learn', 'PyTorch', 'TensorFlow', 'Git', 'GitHub', 'SQL databases', 'Good problem solving skills']


In [13]:
analysis = analyze_requirements(flat_requirements)
print_candidate_report(analysis)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

AI RESUME SCREENING REPORT

MATCH SCORE: 45.45%

MATCHED
------------------------------------------------------------

✓ Python programming
  Source: SKILLS → languages
  Evidence: SKILLS — LANGUAGES
JavaScript, TypeScript, PHP, Python, SQL, HTML, CSS 


✓ Git
  Source: SKILLS → tools
  Evidence: SKILLS — TOOLS
Git, GitHub 
 


✓ GitHub
  Source: SKILLS → tools
  Evidence: SKILLS — TOOLS
Git, GitHub 
 


✓ SQL databases
  Source: SKILLS → languages
  Evidence: SKILLS — LANGUAGES
JavaScript, TypeScript, PHP, Python, SQL, HTML, CSS 


✓ Good problem solving skills
  Source: SKILLS → soft_skills
  Evidence: SKILLS — SOFT_SKILLS
Time management, problem solving, teamwork, adaptability, attention to detail, 
reliability 
 


MISSING
------------------------------------------------------------

✗ Machine learning algorithms
  No supporting evidence found in the resume.

✗ NLP
  No supporting evidence found in the resume.

✗ Transformers
  No supporting evidence found in the resume.

✗ s

In [14]:
prompt = build_candidate_prompt(analysis)

assessment = generate_candidate_assessment(prompt)

assessment = validate_assessment(
    assessment,
    analysis
)

print(json.dumps(assessment, indent=2))


{
  "overall_assessment": "The candidate's skills are partially matched with the job requirements.",
  "confirmed_strengths": [
    "Python programming",
    "Git",
    "GitHub",
    "SQL databases"
  ],
  "missing_requirements": [
    "Machine learning algorithms",
    "NLP",
    "Transformers",
    "scikit-learn",
    "PyTorch",
    "TensorFlow"
  ],
  "final_recommendation": "The candidate may need to improve their skills in machine learning algorithms, NLP, and other areas not explicitly mentioned in the job requirements."
}


In [15]:
result = check_skill("Python programming")
print(result)

{'skill': 'Python programming', 'normalized_skill': 'Python', 'matched': True, 'evidence': [{'document': 'SKILLS — LANGUAGES\nJavaScript, TypeScript, PHP, Python, SQL, HTML, CSS \n\uf0b7', 'metadata': {'candidate_id': 'candidate_001', 'subsection': 'languages', 'section': 'SKILLS', 'resume_id': 'resume_001'}, 'distance': 1.298758864402771, 'rerank_score': -1.405313491821289}]}


In [16]:
result = check_skill("Git")
print(result)

{'skill': 'Git', 'normalized_skill': 'Git', 'matched': True, 'evidence': [{'document': 'SKILLS — TOOLS\nGit, GitHub \n \n\uf0b7', 'metadata': {'section': 'SKILLS', 'subsection': 'tools', 'candidate_id': 'candidate_001', 'resume_id': 'resume_001'}, 'distance': 0.7148358821868896, 'rerank_score': 2.1593990325927734}]}


In [17]:
result = check_skill("Good problem solving skills")
print(result)

{'skill': 'Good problem solving skills', 'normalized_skill': 'Problem solving', 'matched': True, 'evidence': [{'document': 'SKILLS — SOFT_SKILLS\nTime management, problem solving, teamwork, adaptability, attention to detail, \nreliability \n \n\uf0b7', 'metadata': {'section': 'SKILLS', 'candidate_id': 'candidate_001', 'resume_id': 'resume_001', 'subsection': 'soft_skills'}, 'distance': 1.379209041595459, 'rerank_score': 1.3276112079620361}]}


In [18]:
# Quick sanity check on the bug that started all this
result = check_skill("SQL databases")
print(result)

{'skill': 'SQL databases', 'normalized_skill': 'SQL', 'matched': True, 'evidence': [{'document': 'SKILLS — LANGUAGES\nJavaScript, TypeScript, PHP, Python, SQL, HTML, CSS \n\uf0b7', 'metadata': {'resume_id': 'resume_001', 'subsection': 'languages', 'candidate_id': 'candidate_001', 'section': 'SKILLS'}, 'distance': 1.4565991163253784, 'rerank_score': -3.397836208343506}]}


In [19]:
test_skills = [
    "Python programming",
    "React",
    "Laravel",
    "MySQL",
    "SQL databases",
    "Git",
    "Good problem solving skills",
    "NLP",
    "PyTorch",
    "TensorFlow",
]

for skill in test_skills:
    result = check_skill(skill)

    print("=" * 60)
    print("SKILL:", skill)
    print("MATCHED:", result["matched"])

    for evidence in result.get("evidence", []):
        metadata = evidence["metadata"]
        print(
            "SOURCE:",
            metadata.get("section"),
            "→",
            metadata.get("subsection")
        )

SKILL: Python programming
MATCHED: True
SOURCE: SKILLS → languages
SKILL: React
MATCHED: True
SOURCE: ABOUT ME → None
SOURCE: SKILLS → frontend
SKILL: Laravel
MATCHED: True
SOURCE: PROJECTS → None
SOURCE: SKILLS → backend
SKILL: MySQL
MATCHED: True
SOURCE: ABOUT ME → None
SOURCE: SKILLS → database
SKILL: SQL databases
MATCHED: True
SOURCE: SKILLS → languages
SKILL: Git
MATCHED: True
SOURCE: SKILLS → tools
SKILL: Good problem solving skills
MATCHED: True
SOURCE: SKILLS → soft_skills
SKILL: NLP
MATCHED: False
SKILL: PyTorch
MATCHED: False
SKILL: TensorFlow
MATCHED: False


## 8. Robustness check

Sanity check on a fixed, hand-picked requirement list (no LLM extraction step, so results are fully deterministic run-to-run). This is the baseline the multi-candidate ranking feature will build on.

In [20]:
requirements = [
    "Python",
    "Machine learning algorithms",
    "NLP",
    "Transformers",
    "scikit-learn",
    "PyTorch",
    "TensorFlow",
    "SQL databases",
    "Git",
    "GitHub",
    "Good problem solving skills"
]

report = build_match_report(requirements)

print("MATCH SCORE:", report["score"])

print("\nMATCHED")
for item in report["matched"]:
    print("\u2713", item["skill"])

print("\nMISSING")
for item in report["missing"]:
    print("\u2717", item["skill"])


MATCH SCORE: 45.45

MATCHED
✓ Python
✓ SQL databases
✓ Git
✓ GitHub
✓ Good problem solving skills

MISSING
✗ Machine learning algorithms
✗ NLP
✗ Transformers
✗ scikit-learn
✗ PyTorch
✗ TensorFlow


## Assessment Generation

The final assessment is generated from verified matched/missing
requirements only.

The LLM returns structured JSON with:
- overall_assessment
- confirmed_strengths
- missing_requirements
- final_recommendation

The model is not responsible for calculating the match score or
determining whether a requirement matches the resume.

---
**Note on `RERANK_THRESHOLD`:** `-10.3` was reverse-engineered from one exploratory run comparing "positive" vs "negative" queries in the original notebook. It's model- and data-specific -- if you change the reranker, the resume content, or the phrasing of `check_skill`'s query template, re-run that positive/negative probe and re-pick the threshold rather than trusting this constant blindly.